# Experiment 2: Email Spam/Ham Classification using Naive Bayes and KNN

```
experiment2_spambase.py
========================
ICS1512 - Machine Learning Algorithms Laboratory
Experiment 2: Email Spam/Ham Classification using Naive Bayes and KNN

Uses the reusable module ml_lab_utils.py (from Experiment 1) for:
    - EDA                         -> generate_eda_summary()
    - Classification train/eval   -> train_evaluate_classification()
    - Classification metrics      -> classification_performance_metrics()
    - Global plot style           -> set_plot_style()

Dataset: Spambase (UCI ML Repository / Kaggle mirror), 4601 emails x 57
features + binary target (1 = spam, 0 = ham).
```

## Reusable utilities (`ml_lab_utils`, from Experiment 1)

Inlined here so this notebook runs on its own without a separate `ml_lab_utils.py`.

In [ ]:
"""
ml_lab_utils.py
================
ICS1512 - Machine Learning Algorithms Laboratory
Reusable utility module used across ALL experiments.

Implements (per lab manual, Section 4):
    1. One reusable EDA function            -> generate_eda_summary()
    2. One reusable Regression function      -> train_evaluate_regression()
    3. One reusable Classification function  -> train_evaluate_classification()
    4. One reusable Regression metrics fn    -> regression_performance_metrics()
    5. One reusable Classification metrics   -> classification_performance_metrics()

Formatting rules enforced everywhere (per lab manual, Section 1):
    - Times New Roman, 15 pt for all text / legends
    - Bold, Times New Roman, 15 pt axis labels
    - Figures exported as .eps at 600 DPI (Section 3)

NOTE on fonts: "Times New Roman" itself is a proprietary Microsoft font and is
not installable on Linux. Liberation Serif is metrically-compatible (identical
glyph widths/kerning) and is registered here under the family name
"Times New Roman" so that rcParams['font.family'] = 'Times New Roman' works
transparently. On Windows/macOS, if the real Times New Roman is installed,
matplotlib will simply use that instead.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# 1. GLOBAL PLOT STYLE  (Section 1 of the manual)
# --------------------------------------------------------------------------
def set_plot_style(font_size=15):
    """
    Applies the mandatory lab formatting to every matplotlib figure:
        - Times New Roman (or metric-compatible Liberation Serif) font
        - 15 pt base font size
        - 15 pt Times New Roman legends
        - Bold, 15 pt, Times New Roman axis labels
    Call this once at the start of a notebook / script.
    """
    # Register Liberation Serif under the alias "Times New Roman" if the
    # genuine font is not present on this machine.
    installed_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in installed_fonts:
        liberation_paths = [
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf",
        ]
        for p in liberation_paths:
            if os.path.exists(p):
                fm.fontManager.addfont(p)
                # Force the registered family name to "Times New Roman"
                # (FontEntry is a frozen dataclass in modern matplotlib, so we
                # replace the last-added entry rather than mutate it in place)
                last = fm.fontManager.ttflist[-1]
                fm.fontManager.ttflist[-1] = fm.FontEntry(
                    fname=last.fname, name="Times New Roman",
                    style=last.style, variant=last.variant,
                    weight=last.weight, stretch=last.stretch, size=last.size,
                )

    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "axes.labelsize": font_size,
        "axes.labelweight": "bold",
        "axes.titlesize": font_size,
        "axes.titleweight": "bold",
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
        "figure.titlesize": font_size + 2,
        "savefig.dpi": 600,
        "figure.dpi": 150,   # screen preview; export always forced to 600 (see save)
        "svg.fonttype": "none",
    })


def _bold_axis_labels(ax, xlabel=None, ylabel=None, title=None, fs=15):
    """Helper: apply Times New Roman / Bold / 15pt to a single axis explicitly."""
    fp_bold = fm.FontProperties(family="Times New Roman", weight="bold", size=fs)
    fp_reg = fm.FontProperties(family="Times New Roman", size=fs - 2)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=fp_bold)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=fp_bold)
    if title is not None:
        ax.set_title(title, fontproperties=fp_bold, fontsize=fs)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(fp_reg)
    leg = ax.get_legend()
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontproperties(fp_reg)


def _save_eps(fig, save_path, also_png=True):
    """Export a figure as .eps at 600 DPI (Section 3 of the manual).

    If also_png is True, an additional .png copy is saved alongside the .eps
    (same basename) purely so the figure can be embedded when compiling the
    LaTeX report with pdflatex/xelatex, which cannot rasterize .eps directly
    without Ghostscript. The .eps remains the official, mandated deliverable.
    """
    if save_path is None:
        return None
    if not save_path.lower().endswith(".eps"):
        save_path = os.path.splitext(save_path)[0] + ".eps"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, format="eps", dpi=600, bbox_inches="tight")
    if also_png:
        png_path = os.path.splitext(save_path)[0] + ".png"
        fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
    return save_path


# --------------------------------------------------------------------------
# 2. GENERIC EDA FUNCTION  (Section 4.1)  -> ONE consolidated 12-subplot figure
# --------------------------------------------------------------------------
def generate_eda_summary(df, target_col=None, dataset_name="Dataset",
                          save_path=None, figsize=(22, 16)):
    """
    Generic, reusable EDA function that works on ANY tabular dataset
    (classification, regression, or unlabeled). Produces ONE consolidated
    figure containing 12 EDA subplots on a single page, per Section 2 of the
    lab manual.

    Parameters
    ----------
    df : pandas.DataFrame
        The full dataset (features + target, if any).
    target_col : str or None
        Name of the target/label column, if present. If None, the function
        treats the dataset as unlabeled and adapts the 12-panel layout
        accordingly (no class-distribution / target-correlation panels).
    dataset_name : str
        Used in the figure's suptitle.
    save_path : str or None
        If given, the figure is exported as .eps @ 600 DPI to this path.
    figsize : tuple
        Overall figure size in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    set_plot_style()
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    is_classification_target = (
        target_col is not None and
        (df[target_col].dtype == "object" or df[target_col].nunique() <= 20)
    )

    # Pick the most "informative" numeric feature (highest variance) as the
    # representative single feature for panels 6/8/9/10, instead of blindly
    # using the first column (which can be degenerate/constant, e.g. corner
    # pixels in an image dataset such as MNIST/Digits).
    if numeric_cols:
        # Prefer genuinely continuous columns (more than 5 distinct values) so
        # binary/near-constant encoded columns (e.g. a 0/1 "sex" flag, or
        # constant corner pixels in image data) are not picked as the
        # representative single feature for panels 6/8/9/10.
        continuous_cols = [c for c in numeric_cols if df[c].nunique() > 5]
        candidate_cols = continuous_cols if continuous_cols else numeric_cols
        variances = df[candidate_cols].var().sort_values(ascending=False)
        top_var_cols = variances.index.tolist()
        feat_a = top_var_cols[0]
        feat_b = top_var_cols[1] if len(top_var_cols) > 1 else top_var_cols[0]
        kde_cols = top_var_cols[:4]
    else:
        feat_a = feat_b = None
        kde_cols = []

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"Exploratory Data Analysis Summary \u2013 {dataset_name}",
                 fontweight="bold", fontsize=17,
                 fontproperties=fm.FontProperties(family="Times New Roman",
                                                   weight="bold", size=17))
    gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)
    axes = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(12)]
    panel = 0

    # ---- Panel 1: Dataset overview (head / shape as a text table) ----
    ax = axes[panel]; panel += 1
    ax.axis("off")
    overview_txt = (
        f"Shape: {df.shape[0]} rows x {df.shape[1]} cols\n"
        f"Numeric features: {len(numeric_cols)}\n"
        f"Categorical features: {len(categorical_cols)}\n"
        f"Missing cells: {int(df.isnull().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}"
    )
    ax.text(0.02, 0.9, overview_txt, va="top", ha="left",
            fontproperties=fm.FontProperties(family="Times New Roman", size=13),
            transform=ax.transAxes)
    _bold_axis_labels(ax, title="1. Dataset Overview")

    # ---- Panel 2: Statistical summary heat-table (mean/std/min/max) ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[["mean", "std", "min", "max"]]
        desc_norm = (desc - desc.min()) / (desc.max() - desc.min() + 1e-9)
        sns.heatmap(desc_norm.iloc[:8], annot=desc.iloc[:8].round(1), fmt="",
                    cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontsize": 8, "fontfamily": "Times New Roman"})
    _bold_axis_labels(ax, title="2. Statistical Summary")

    # ---- Panel 3: Missing value analysis ----
    ax = axes[panel]; panel += 1
    miss = df.isnull().mean().sort_values(ascending=False) * 100
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=14))
        ax.axis("off")
    else:
        miss[miss > 0].head(10).plot(kind="bar", ax=ax, color="#c0392b")
    _bold_axis_labels(ax, "Feature", "% Missing", "3. Missing Value Analysis")

    # ---- Panel 4: Class distribution / target distribution ----
    ax = axes[panel]; panel += 1
    if target_col is not None:
        if is_classification_target:
            df[target_col].value_counts().plot(kind="bar", ax=ax, color="#2980b9")
            _bold_axis_labels(ax, "Class", "Count", "4. Class Distribution")
        else:
            sns.histplot(df[target_col], kde=True, ax=ax, color="#2980b9")
            _bold_axis_labels(ax, target_col, "Frequency", "4. Target Distribution")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "No target column supplied", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=12))
        _bold_axis_labels(ax, title="4. Target Distribution")

    # ---- Panel 5: Correlation matrix (heatmap) ----
    ax = axes[panel]; panel += 1
    corr_cols = numeric_cols[:10] if len(numeric_cols) > 10 else numeric_cols
    if len(corr_cols) >= 2:
        sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, ax=ax,
                    cbar=False, annot=len(corr_cols) <= 6, fmt=".2f",
                    annot_kws={"fontsize": 7})
    _bold_axis_labels(ax, title="5. Correlation Matrix")

    # ---- Panel 6: Feature distribution (histogram of 1st numeric feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.histplot(df[feat_a], kde=True, ax=ax, color="#27ae60")
    _bold_axis_labels(ax, feat_a if feat_a else "", "Frequency",
                       "6. Feature Distribution")

    # ---- Panel 7: Box plot (outlier detection) across numeric features ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        plot_cols = numeric_cols[:6]
        df_scaled = (df[plot_cols] - df[plot_cols].mean()) / (df[plot_cols].std() + 1e-9)
        sns.boxplot(data=df_scaled, ax=ax, color="#f39c12")
        ax.tick_params(axis="x", rotation=45)
    _bold_axis_labels(ax, "Feature", "Standardized Value", "7. Box Plot (Outliers)")

    # ---- Panel 8: Violin plot ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.violinplot(y=df[feat_a], ax=ax, color="#8e44ad")
    _bold_axis_labels(ax, "", feat_a if feat_a else "",
                       "8. Violin Plot")

    # ---- Panel 9: Scatter plot (feature 1 vs feature 2, hued by target) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None and feat_b is not None:
        hue = df[target_col] if (target_col and is_classification_target) else None
        sns.scatterplot(x=df[feat_a], y=df[feat_b],
                         hue=hue, ax=ax, palette="Set2", legend=False, s=18)
    _bold_axis_labels(ax, feat_a if feat_a else "", feat_b if feat_b else "",
                       "9. Scatter Plot")

    # ---- Panel 10: Q-Q plot (normality check on highest-variance feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        stats.probplot(df[feat_a].dropna(), dist="norm", plot=ax)
        ax.get_lines()[0].set_markerfacecolor("#2980b9")
        ax.get_lines()[0].set_markeredgecolor("#2980b9")
        ax.get_lines()[1].set_color("#c0392b")
    _bold_axis_labels(ax, "Theoretical Quantiles", "Sample Quantiles", "10. Q-Q Plot")

    # ---- Panel 11: KDE / density plot overlay of top numeric features ----
    ax = axes[panel]; panel += 1
    for c in kde_cols:
        sns.kdeplot(df[c], ax=ax, label=c, linewidth=1.5)
    if kde_cols:
        ax.legend(prop=fm.FontProperties(family="Times New Roman", size=9))
    _bold_axis_labels(ax, "Value", "Density", "11. KDE / Density Plot")

    # ---- Panel 12: Feature importance / variance plot ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        var = df[numeric_cols].var().sort_values(ascending=False).head(8)
        var.plot(kind="barh", ax=ax, color="#16a085")
        ax.invert_yaxis()
    _bold_axis_labels(ax, "Variance", "Feature", "12. Variance / Importance Plot")

    for ax in axes:
        _bold_axis_labels(ax)  # re-apply tick font in case a plotting call reset it

    saved = _save_eps(fig, save_path)
    if saved:
        print(f"[generate_eda_summary] Figure saved -> {saved} (600 DPI, EPS)")
    return fig


# --------------------------------------------------------------------------
# 3. GENERIC REGRESSION TRAIN/EVAL FUNCTION  (Section 4.2)
# --------------------------------------------------------------------------
def train_evaluate_regression(models: dict, X_train, X_test, y_train, y_test,
                               scale=False, verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of regression
    models on the same train/test split.

    Parameters
    ----------
    models : dict {name: sklearn-estimator}
    X_train, X_test, y_train, y_test : array-like
    scale : bool -> StandardScaler applied when True (fit on train only)
    verbose : bool -> print per-model metrics as they are computed

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by R2 desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                                   verbose=verbose, return_dict=True)
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 4. GENERIC CLASSIFICATION TRAIN/EVAL FUNCTION  (Section 4.3)
# --------------------------------------------------------------------------
def train_evaluate_classification(models: dict, X_train, X_test, y_train, y_test,
                                   scale=False, average="weighted", verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of classification
    models on the same train/test split.

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by Accuracy desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = None
        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)
            except Exception:
                y_proba = None
        metrics = classification_performance_metrics(
            y_test, y_pred, y_proba=y_proba, model_name=name,
            average=average, verbose=verbose, return_dict=True, plot=False
        )
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 5. GENERIC REGRESSION METRICS FUNCTION  (Section 4.4)
# --------------------------------------------------------------------------
def regression_performance_metrics(y_true, y_pred, model_name="Model",
                                    verbose=True, return_dict=False):
    """
    Computes and displays ALL standard regression performance metrics:
    MAE, MSE, RMSE, R2, Adjusted R2 (n only), MAPE.
    """
    from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                  r2_score, mean_absolute_percentage_error)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    if verbose:
        print(f"--- Regression Metrics: {model_name} ---")
        print(f"  MAE  : {mae:.4f}")
        print(f"  MSE  : {mse:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R2   : {r2:.4f}")
        print(f"  MAPE : {mape:.2f}%\n")

    result = {"Model": model_name, "MAE": mae, "MSE": mse,
              "RMSE": rmse, "R2": r2, "MAPE(%)": mape}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


# --------------------------------------------------------------------------
# 6. GENERIC CLASSIFICATION METRICS FUNCTION  (Section 4.5)
# --------------------------------------------------------------------------
def classification_performance_metrics(y_true, y_pred, y_proba=None,
                                        model_name="Model", average="weighted",
                                        verbose=True, return_dict=False,
                                        plot=True, save_path=None):
    """
    Computes and displays ALL standard classification performance metrics:
    Accuracy, Precision, Recall, F1-score, ROC-AUC (binary/multiclass ovr),
    and (optionally) plots the confusion matrix using the mandatory lab
    formatting (Times New Roman, bold 15pt axis labels).
    """
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score, confusion_matrix)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)

    roc_auc = np.nan
    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                roc_auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr",
                                         average=average)
        except Exception:
            roc_auc = np.nan

    if verbose:
        print(f"--- Classification Metrics: {model_name} ---")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC   : N/A")
        print()

    if plot:
        set_plot_style()
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontfamily": "Times New Roman", "fontsize": 13})
        _bold_axis_labels(ax, "Predicted Label", "True Label",
                           f"Confusion Matrix \u2013 {model_name}")
        _save_eps(fig, save_path)

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-score": f1, "ROC-AUC": roc_auc}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


In [1]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      RandomizedSearchCV, StratifiedKFold,
                                      cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              precision_recall_curve, confusion_matrix)


warnings.filterwarnings("ignore")
RANDOM_STATE = 42

FIG_DIR = "figures"
RES_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

set_plot_style()


## 1. LOAD DATASET

In [2]:
COLUMN_NAMES = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d",
    "word_freq_our", "word_freq_over", "word_freq_remove", "word_freq_internet",
    "word_freq_order", "word_freq_mail", "word_freq_receive", "word_freq_will",
    "word_freq_people", "word_freq_report", "word_freq_addresses", "word_freq_free",
    "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money",
    "word_freq_hp", "word_freq_hpl", "word_freq_george", "word_freq_650",
    "word_freq_lab", "word_freq_labs", "word_freq_telnet", "word_freq_857",
    "word_freq_data", "word_freq_415", "word_freq_85", "word_freq_technology",
    "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project",
    "word_freq_re", "word_freq_edu", "word_freq_table", "word_freq_conference",
    "char_freq_semicolon", "char_freq_paren", "char_freq_bracket",
    "char_freq_bang", "char_freq_dollar", "char_freq_pound",
    "capital_run_length_average", "capital_run_length_longest",
    "capital_run_length_total", "spam",
]

DATA_PATH = "spambase.csv"  # <-- upload this file to the notebook's working directory
raw = pd.read_csv(DATA_PATH)
raw.columns = COLUMN_NAMES
df = raw.copy()
print("Dataset shape:", df.shape)
print(df["spam"].value_counts())

Dataset shape: (4601, 58)
spam
0    2788
1    1813
Name: count, dtype: int64


## 2. HANDLE MISSING VALUES (none expected, verified defensively)

In [3]:
n_missing = int(df.isnull().sum().sum())
print(f"Missing cells: {n_missing}")
if n_missing > 0:
    df = df.fillna(df.median(numeric_only=True))

Missing cells: 0


## 3. EDA  (reusable function from Experiment 1)

In [4]:
generate_eda_summary(
    df, target_col="spam", dataset_name="Spambase",
    save_path=f"{FIG_DIR}/eda_spambase.eps"
)
plt.close("all")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[generate_eda_summary] Figure saved -> figures/eda_spambase.eps (600 DPI, EPS)


## 4. TRAIN / TEST SPLIT + FEATURE SCALING

In [5]:
X = df.drop(columns=["spam"]).values
y = df["spam"].values
feature_names = df.drop(columns=["spam"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Non-negative version for MultinomialNB (which requires X >= 0); StandardScaler
# output has negative values, so Multinomial NB uses the raw (unscaled) features.
X_train_raw, X_test_raw = X_train, X_test

## 5. NAIVE BAYES: GAUSSIAN, MULTINOMIAL, BERNOULLI

In [6]:
nb_results = []
nb_models = {}
nb_timings = {}

def time_fit_predict(model, Xtr, Xte, ytr):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    train_t = time.perf_counter() - t0
    t0 = time.perf_counter()
    y_pred = model.predict(Xte)
    pred_t = time.perf_counter() - t0
    return y_pred, train_t, pred_t

# Gaussian NB -> scaled continuous features
gnb = GaussianNB()
y_pred, train_t, pred_t = time_fit_predict(gnb, X_train_scaled, X_test_scaled, y_train)
y_proba = gnb.predict_proba(X_test_scaled)
m = classification_performance_metrics(y_test, y_pred, y_proba, model_name="Gaussian NB",
                                        return_dict=True, plot=True,
                                        save_path=f"{FIG_DIR}/cm_gaussian_nb.eps")
m.update({"Train Time (s)": train_t, "Predict Time (s)": pred_t})
nb_results.append(m)
nb_models["Gaussian NB"] = gnb
nb_timings["Gaussian NB"] = (train_t, pred_t)

# Multinomial NB -> raw non-negative word/char frequency features
mnb = MultinomialNB()
y_pred, train_t, pred_t = time_fit_predict(mnb, X_train_raw, X_test_raw, y_train)
y_proba = mnb.predict_proba(X_test_raw)
m = classification_performance_metrics(y_test, y_pred, y_proba, model_name="Multinomial NB",
                                        return_dict=True, plot=True,
                                        save_path=f"{FIG_DIR}/cm_multinomial_nb.eps")
m.update({"Train Time (s)": train_t, "Predict Time (s)": pred_t})
nb_results.append(m)
nb_models["Multinomial NB"] = mnb
nb_timings["Multinomial NB"] = (train_t, pred_t)

# Bernoulli NB -> binarized presence/absence of features (threshold 0)
bnb = BernoulliNB()
y_pred, train_t, pred_t = time_fit_predict(bnb, X_train_raw, X_test_raw, y_train)
y_proba = bnb.predict_proba(X_test_raw)
m = classification_performance_metrics(y_test, y_pred, y_proba, model_name="Bernoulli NB",
                                        return_dict=True, plot=True,
                                        save_path=f"{FIG_DIR}/cm_bernoulli_nb.eps")
m.update({"Train Time (s)": train_t, "Predict Time (s)": pred_t})
nb_results.append(m)
nb_models["Bernoulli NB"] = bnb
nb_timings["Bernoulli NB"] = (train_t, pred_t)

nb_results_df = pd.DataFrame(nb_results).set_index("Model")
nb_results_df.to_csv(f"{RES_DIR}/naive_bayes_comparison.csv")
print("\n=== Naive Bayes Comparison ===")
print(nb_results_df)

best_nb_name = nb_results_df["Accuracy"].idxmax()
best_nb_model = nb_models[best_nb_name]
best_nb_X_train = X_train_scaled if best_nb_name == "Gaussian NB" else X_train_raw
best_nb_X_test = X_test_scaled if best_nb_name == "Gaussian NB" else X_test_raw
print(f"\nBest Naive Bayes variant: {best_nb_name}")

--- Classification Metrics: Gaussian NB ---
  Accuracy  : 0.8328
  Precision : 0.8666
  Recall    : 0.8328
  F1-score  : 0.8345
  ROC-AUC   : 0.9376

--- Classification Metrics: Multinomial NB ---
  Accuracy  : 0.7763
  Precision : 0.7757
  Recall    : 0.7763
  F1-score  : 0.7760
  ROC-AUC   : 0.8248



--- Classification Metrics: Bernoulli NB ---
  Accuracy  : 0.8762
  Precision : 0.8760
  Recall    : 0.8762
  F1-score  : 0.8753
  ROC-AUC   : 0.9496




=== Naive Bayes Comparison ===
                Accuracy  Precision    Recall  F1-score   ROC-AUC  \
Model                                                               
Gaussian NB     0.832790   0.866565  0.832790  0.834536  0.937612   
Multinomial NB  0.776330   0.775730  0.776330  0.775996  0.824812   
Bernoulli NB    0.876221   0.876003  0.876221  0.875254  0.949646   

                Train Time (s)  Predict Time (s)  
Model                                             
Gaussian NB           0.004708          0.000699  
Multinomial NB        0.001643          0.000200  
Bernoulli NB          0.003097          0.000777  

Best Naive Bayes variant: Bernoulli NB


## 6. KNN: EFFECT OF VARYING k

In [7]:
k_values = [1, 3, 5, 7, 9, 11]
knn_k_results = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    knn_k_results.append({
        "k": k,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    })
knn_k_df = pd.DataFrame(knn_k_results).set_index("k")
knn_k_df.to_csv(f"{RES_DIR}/knn_k_comparison.csv")
print("\n=== KNN: Accuracy vs k ===")
print(knn_k_df)

best_k = knn_k_df["Accuracy"].idxmax()
print(f"Best k (plain KNN sweep): {best_k}")

# Plot: Accuracy vs k
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(knn_k_df.index, knn_k_df["Accuracy"], marker="o", color="#2980b9", linewidth=2)
_bold_axis_labels(ax, "k (Number of Neighbors)", "Accuracy", "Accuracy vs k (KNN)")
_save_eps(fig, f"{FIG_DIR}/accuracy_vs_k.eps")
plt.close(fig)


=== KNN: Accuracy vs k ===
    Accuracy  Precision    Recall        F1
k                                          
1   0.899023   0.879213  0.862259  0.870654
3   0.897937   0.872576  0.867769  0.870166
5   0.907709   0.886111  0.878788  0.882434
7   0.908795   0.892958  0.873278  0.883008
9   0.908795   0.892958  0.873278  0.883008
11  0.909881   0.900000  0.867769  0.883590
Best k (plain KNN sweep): 11


## 7. GRIDSEARCHCV vs RANDOMIZEDSEARCHCV FOR KNN

In [8]:
param_grid = {
    "n_neighbors": [1, 3, 5, 7, 9, 11, 13, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
    "algorithm": ["kd_tree", "ball_tree"],
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

t0 = time.perf_counter()
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=cv_strategy,
                            scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)
grid_time = time.perf_counter() - t0

t0 = time.perf_counter()
random_search = RandomizedSearchCV(KNeighborsClassifier(), param_grid, cv=cv_strategy,
                                    scoring="accuracy", n_iter=20,
                                    random_state=RANDOM_STATE, n_jobs=-1)
random_search.fit(X_train_scaled, y_train)
random_time = time.perf_counter() - t0

search_comparison = pd.DataFrame({
    "GridSearchCV": {
        "Best k": grid_search.best_params_["n_neighbors"],
        "Metric": grid_search.best_params_["metric"],
        "Weights": grid_search.best_params_["weights"],
        "Algorithm": grid_search.best_params_["algorithm"],
        "CV Accuracy": grid_search.best_score_,
        "Execution Time (s)": grid_time,
    },
    "RandomizedSearchCV": {
        "Best k": random_search.best_params_["n_neighbors"],
        "Metric": random_search.best_params_["metric"],
        "Weights": random_search.best_params_["weights"],
        "Algorithm": random_search.best_params_["algorithm"],
        "CV Accuracy": random_search.best_score_,
        "Execution Time (s)": random_time,
    },
})
search_comparison.to_csv(f"{RES_DIR}/gridsearch_vs_randomsearch.csv")
print("\n=== GridSearchCV vs RandomizedSearchCV ===")
print(search_comparison)

best_knn_params = grid_search.best_params_
best_knn = KNeighborsClassifier(**best_knn_params)
best_knn.fit(X_train_scaled, y_train)
y_pred_best_knn = best_knn.predict(X_test_scaled)
y_proba_best_knn = best_knn.predict_proba(X_test_scaled)

best_knn_metrics = classification_performance_metrics(
    y_test, y_pred_best_knn, y_proba_best_knn, model_name="Best KNN (Tuned)",
    return_dict=True, plot=True, save_path=f"{FIG_DIR}/cm_best_knn.eps"
)
print("\n=== Best (Tuned) KNN Metrics ===")
print(best_knn_metrics)

# GridSearchCV heatmap (k vs metric, mean of weights/algorithm)
cv_results = pd.DataFrame(grid_search.cv_results_)
pivot = cv_results.pivot_table(values="mean_test_score",
                                index="param_n_neighbors",
                                columns="param_metric", aggfunc="mean")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", ax=ax,
            annot_kws={"fontfamily": "Times New Roman", "fontsize": 10})
_bold_axis_labels(ax, "Distance Metric", "k (n_neighbors)", "GridSearchCV Heatmap")
_save_eps(fig, f"{FIG_DIR}/gridsearch_heatmap.eps")
plt.close(fig)

# RandomizedSearchCV score distribution
random_cv_results = pd.DataFrame(random_search.cv_results_)
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(random_cv_results["mean_test_score"], kde=True, ax=ax, color="#8e44ad")
_bold_axis_labels(ax, "Mean CV Accuracy", "Frequency",
                   "RandomizedSearchCV Score Distribution")
_save_eps(fig, f"{FIG_DIR}/randomsearch_score_distribution.eps")
plt.close(fig)


=== GridSearchCV vs RandomizedSearchCV ===
                   GridSearchCV RandomizedSearchCV
Best k                        9                  9
Metric                manhattan          manhattan
Weights                distance           distance
Algorithm               kd_tree            kd_tree
CV Accuracy            0.925272           0.925272
Execution Time (s)    26.983828           8.242071


--- Classification Metrics: Best KNN (Tuned) ---
  Accuracy  : 0.9207
  Precision : 0.9220
  Recall    : 0.9207
  F1-score  : 0.9199
  ROC-AUC   : 0.9712


=== Best (Tuned) KNN Metrics ===
{'Model': 'Best KNN (Tuned)', 'Accuracy': 0.9207383279044516, 'Precision': 0.9219975513428226, 'Recall': 0.9207383279044516, 'F1-score': 0.9199360852684387, 'ROC-AUC': 0.9712126149076296}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


## 8. KDTREE vs BALLTREE

In [9]:
tree_comparison = {}
for algo in ["kd_tree", "ball_tree"]:
    knn_tree = KNeighborsClassifier(n_neighbors=best_knn_params["n_neighbors"],
                                     weights=best_knn_params["weights"],
                                     metric=best_knn_params["metric"],
                                     algorithm=algo)
    y_pred, train_t, pred_t = time_fit_predict(knn_tree, X_train_scaled, X_test_scaled, y_train)
    tree_comparison[algo] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Training Time (s)": train_t,
        "Prediction Time (s)": pred_t,
    }
tree_comparison_df = pd.DataFrame(tree_comparison)
tree_comparison_df.to_csv(f"{RES_DIR}/kdtree_vs_balltree.csv")
print("\n=== KDTree vs BallTree ===")
print(tree_comparison_df)


=== KDTree vs BallTree ===
                      kd_tree  ball_tree
Accuracy             0.920738   0.920738
Training Time (s)    0.011494   0.006855
Prediction Time (s)  0.141057   0.111816


## 9. 5-FOLD CROSS VALIDATION (Naive Bayes best vs Best KNN)

In [10]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_scaled_full = scaler.fit_transform(X)  # refit scaler on full X for CV convenience
nb_cv_scores = cross_val_score(GaussianNB() if best_nb_name == "Gaussian NB" else
                                (MultinomialNB() if best_nb_name == "Multinomial NB" else BernoulliNB()),
                                X if best_nb_name != "Gaussian NB" else X_scaled_full,
                                y, cv=skf, scoring="accuracy")
knn_cv_scores = cross_val_score(KNeighborsClassifier(**best_knn_params),
                                 X_scaled_full, y, cv=skf, scoring="accuracy")

cv_table = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)] + ["Average"],
    f"Naive Bayes ({best_nb_name})": list(nb_cv_scores) + [nb_cv_scores.mean()],
    "Best KNN": list(knn_cv_scores) + [knn_cv_scores.mean()],
}).set_index("Fold")
cv_table.to_csv(f"{RES_DIR}/cross_validation.csv")
print("\n=== 5-Fold Cross Validation ===")
print(cv_table)

fig, ax = plt.subplots(figsize=(7, 5))
folds = np.arange(1, 6)
ax.plot(folds, nb_cv_scores, marker="o", label=f"Naive Bayes ({best_nb_name})", color="#c0392b")
ax.plot(folds, knn_cv_scores, marker="s", label="Best KNN", color="#2980b9")
ax.legend()
_bold_axis_labels(ax, "Fold", "Accuracy", "Cross-Validation Accuracy")
_save_eps(fig, f"{FIG_DIR}/cv_accuracy.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== 5-Fold Cross Validation ===
         Naive Bayes (Bernoulli NB)  Best KNN
Fold                                         
Fold 1                     0.884908  0.916395
Fold 2                     0.885870  0.932609
Fold 3                     0.879348  0.935870
Fold 4                     0.902174  0.921739
Fold 5                     0.881522  0.921739
Average                    0.886764  0.925670


## 10. THEORETICAL vs EXPERIMENTAL TIME COMPLEXITY

In [11]:
theoretical_complexity = pd.DataFrame({
    "Algorithm": ["Naive Bayes", "KNN (Brute)", "KDTree", "BallTree"],
    "Training": ["O(nd)", "O(1)", "O(n log n)", "O(n log n)"],
    "Prediction": ["O(d)", "O(nd)", "O(log n) avg", "O(log n) avg"],
}).set_index("Algorithm")
theoretical_complexity.to_csv(f"{RES_DIR}/theoretical_complexity.csv")

experimental_time = pd.DataFrame({
    "Gaussian NB": {"Training(s)": nb_timings["Gaussian NB"][0],
                    "Prediction(s)": nb_timings["Gaussian NB"][1]},
    "Multinomial NB": {"Training(s)": nb_timings["Multinomial NB"][0],
                       "Prediction(s)": nb_timings["Multinomial NB"][1]},
    "Bernoulli NB": {"Training(s)": nb_timings["Bernoulli NB"][0],
                     "Prediction(s)": nb_timings["Bernoulli NB"][1]},
}).T
experimental_time.to_csv(f"{RES_DIR}/experimental_time.csv")
print("\n=== Experimental Time Analysis (Naive Bayes) ===")
print(experimental_time)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
experimental_time["Training(s)"].plot(kind="bar", ax=axes[0], color="#16a085")
_bold_axis_labels(axes[0], "Model", "Time (s)", "Training Time Comparison")
experimental_time["Prediction(s)"].plot(kind="bar", ax=axes[1], color="#e67e22")
_bold_axis_labels(axes[1], "Model", "Time (s)", "Prediction Time Comparison")
plt.tight_layout()
_save_eps(fig, f"{FIG_DIR}/training_prediction_time.eps")
plt.close(fig)


=== Experimental Time Analysis (Naive Bayes) ===
                Training(s)  Prediction(s)
Gaussian NB        0.004708       0.000699
Multinomial NB     0.001643       0.000200
Bernoulli NB       0.003097       0.000777


## 11. ROC CURVES, PRECISION-RECALL CURVES, CLASSIFIER COMPARISON

In [12]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model, Xte in [
    ("Gaussian NB", nb_models["Gaussian NB"], X_test_scaled),
    ("Multinomial NB", nb_models["Multinomial NB"], X_test_raw),
    ("Bernoulli NB", nb_models["Bernoulli NB"], X_test_raw),
    ("Best KNN", best_knn, X_test_scaled),
]:
    proba = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linewidth=1.8)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.legend()
_bold_axis_labels(ax, "False Positive Rate", "True Positive Rate", "ROC Curves")
_save_eps(fig, f"{FIG_DIR}/roc_curves.eps")
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
for name, model, Xte in [
    ("Gaussian NB", nb_models["Gaussian NB"], X_test_scaled),
    ("Multinomial NB", nb_models["Multinomial NB"], X_test_raw),
    ("Bernoulli NB", nb_models["Bernoulli NB"], X_test_raw),
    ("Best KNN", best_knn, X_test_scaled),
]:
    proba = model.predict_proba(Xte)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax.plot(rec, prec, label=name, linewidth=1.8)
ax.legend()
_bold_axis_labels(ax, "Recall", "Precision", "Precision-Recall Curves")
_save_eps(fig, f"{FIG_DIR}/precision_recall_curves.eps")
plt.close(fig)

all_models_summary = pd.concat([
    nb_results_df[["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]],
    pd.DataFrame([best_knn_metrics]).set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]],
])
all_models_summary.to_csv(f"{RES_DIR}/classifier_comparison.csv")
print("\n=== Overall Classifier Comparison ===")
print(all_models_summary)

fig, ax = plt.subplots(figsize=(9, 5))
all_models_summary["Accuracy"].plot(kind="bar", ax=ax, color="#2c3e50")
_bold_axis_labels(ax, "Model", "Accuracy", "Classifier Comparison (Accuracy)")
plt.xticks(rotation=30, ha="right")
_save_eps(fig, f"{FIG_DIR}/classifier_comparison_bar.eps")
plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



=== Overall Classifier Comparison ===
                  Accuracy  Precision    Recall  F1-score   ROC-AUC
Model                                                              
Gaussian NB       0.832790   0.866565  0.832790  0.834536  0.937612
Multinomial NB    0.776330   0.775730  0.776330  0.775996  0.824812
Bernoulli NB      0.876221   0.876003  0.876221  0.875254  0.949646
Best KNN (Tuned)  0.920738   0.921998  0.920738  0.919936  0.971213


## 12. ADDITIONAL TASKS

In [13]:
# (a) Different train-test splits
split_results = []
for test_size in [0.1, 0.2, 0.3, 0.4]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size,
                                           random_state=RANDOM_STATE, stratify=y)
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
    knn_tmp = KNeighborsClassifier(**best_knn_params)
    knn_tmp.fit(Xtr_s, ytr)
    acc = accuracy_score(yte, knn_tmp.predict(Xte_s))
    split_results.append({"Test Size": test_size, "Accuracy": acc})
split_df = pd.DataFrame(split_results).set_index("Test Size")
split_df.to_csv(f"{RES_DIR}/train_test_split_comparison.csv")
print("\n=== Train-Test Split Comparison (Best KNN) ===")
print(split_df)

# (b) Euclidean vs Manhattan distance
dist_results = {}
for metric in ["euclidean", "manhattan"]:
    knn_tmp = KNeighborsClassifier(n_neighbors=best_knn_params["n_neighbors"], metric=metric)
    knn_tmp.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_tmp.predict(X_test_scaled))
    dist_results[metric] = acc
dist_df = pd.DataFrame([dist_results])
dist_df.to_csv(f"{RES_DIR}/distance_metric_comparison.csv")
print("\n=== Euclidean vs Manhattan Distance ===")
print(dist_df)

# (c) Weighted KNN (uniform vs distance)
weight_results = {}
for w in ["uniform", "distance"]:
    knn_tmp = KNeighborsClassifier(n_neighbors=best_knn_params["n_neighbors"], weights=w)
    knn_tmp.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_tmp.predict(X_test_scaled))
    weight_results[w] = acc
weight_df = pd.DataFrame([weight_results])
weight_df.to_csv(f"{RES_DIR}/weighted_knn_comparison.csv")
print("\n=== Weighted KNN Comparison ===")
print(weight_df)

print("\nAll figures saved under:", os.path.abspath(FIG_DIR))
print("All result tables saved under:", os.path.abspath(RES_DIR))
print("\nDone.")


=== Train-Test Split Comparison (Best KNN) ===
           Accuracy
Test Size          
0.1        0.906725
0.2        0.920738
0.3        0.931209
0.4        0.920152

=== Euclidean vs Manhattan Distance ===
   euclidean  manhattan
0   0.908795   0.903366

=== Weighted KNN Comparison ===
    uniform  distance
0  0.908795  0.916395

All figures saved under: /home/claude/notebooks/figures
All result tables saved under: /home/claude/notebooks/results

Done.
